# Gate Status LED Project for 215 Pinehill
Rev. 3
3/2/2021
James

## Objective
1. Permanently on red/green LEDs visible from the house to indicate whether the gate is open or closed without having to squint at the gate.
2. Power lighted gate numerals sign.

## Components
LED assemblies, mounted to gate
Power supply with dimmer, installed into gate control box, powered by gate control AC circuit.

## High Level Diagram

<img src="files/highlevel.png">

### Sources

1. Voltage drop on 10' 18awg @ 10vdc, 1A = `0.13v` [(calculator.net)](https://www.calculator.net/voltage-drop-calculator.html?material=copper&wiresize=20.95&voltage=10&phase=dc&noofconductor=1&distance=10&distanceunit=feet&amperes=1&x=51&y=28)
2. Luxdrive by LEDdynamics [BuckBlock A009 datasheet](https://www.ledsupply.com/content/pdf/led-driver-luxdrive-buckblock_documentation.pdf)

## Design
The CREE LED units do not have current limiting circuitry, so have to use a [constant current supply](https://www.ledsupply.com/blog/understanding-led-drivers/).  The most efficient ones use a separate AC/DC transformer. The A009 unit supports an external dimming potentiometer.

### `VDROP`

Voltage drop on 10' 18awg @ 10vdc, 1A = `0.13v` (source #1)

### `SUPVOLT`
The [BuckBlock DC-DC power supply](https://www.ledsupply.com/content/pdf/led-driver-luxdrive-buckblock_documentation.pdf) requires an external constant voltage source. It measures `51` x `31` x `9.5`mm It needs at least `2`vdc headroom above the theoretical calculation and can deliver `80`% of the voltage supplied to the LEDs. So we have

$$ \textrm{SUPVOLT}= \frac{v_{red}+v_{green}+v_{wire-drop}}{e_{buck-converter}} $$

where $e_{buck-converter}$ is $80\%$ according to the [datasheet](https://www.ledsupply.com/content/pdf/led-driver-luxdrive-buckblock_documentation.pdf) for the 1000mA unit. Therefore:


In [6]:
v_LEDRed = 2.65
v_LEDGreen = 3.7
v_wire = 0.13             # see sources (1), noted as "VDROP" in image
e_buckconverter = 0.80    # see sources (2)

In [5]:
SUPVOLT = round((v_LEDRed+v_LEDGreen+v_wire)/e_buckconverter,1)
SUPVOLT

8.1

so we can a `12`v AC/DC transformer, which is easy to find. We want something that will fit into the project box and be directly wireable into the circuit (zip cord). This CUI Inc. unit ([PSK-S5B-12-T](https://www.cui.com/product/resource/psk-s5b-l.pdf)) is chassis mount with screw terminals. The unit is `76` x `31.5` x `27`mm.

### Potentiometer Selection:

The A009 datasheet says we can use a single `20k`Ω

* [P230-2EC22BR20K](https://www.digikey.com/en/products/detail/tt-electronics-bi/P230-2EC22BR20K/4780751) is the 20k Ω version with solder lugs and panel mount

## PCB Rev. 3

Want to support both off-board A009 LED driver and a new on-board driver which has a different resistor calculation.

* On-Board LED driver: XP Power LDU07 14.0W p/n [LDU1416S1000](https://www.digikey.com/en/products/detail/xp-power/LDU1416S1000) (up to 1000mA, which is sufficient for the red/green LEDs but not the ligted numbers, which can be voltage-driven and have current control built in).

The datasheet says analog dimming is calculated by:

$$I_{out}=\frac{I_{rated max}\cdot R}{(R+200\text{k}\Omega)}$$

In [ ]:
I_max = 1000
R_min = 0
R_max = 2_000_000

#